# Exploration des données

Ce notebook présente les données du challenge et les analyse.

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

In [2]:
# Extraction des données
X = pd.read_csv('../data/raw/x_train_final.csv')
y = pd.read_csv('../data/raw/y_train_final_j5KGWWK.csv')

X.head()

,Unnamed: 0.1,Unnamed: 0,train,gare,date,arret,p2q0,p3q0,p4q0,p0q2,p0q3,p0q4
0,0,0,VBXNMF,KYF,2023-04-03,8,0.0,0.0,1.0,-3.0,-1.0,-2.0
1,1,1,VBXNMF,JLR,2023-04-03,9,0.0,0.0,0.0,1.0,0.0,1.0
2,2,2,VBXNMF,EOH,2023-04-03,10,-1.0,0.0,0.0,-1.0,0.0,0.0
3,3,3,VBXNMF,VXY,2023-04-03,11,-1.0,-1.0,0.0,2.0,-2.0,0.0
4,4,4,VBXNMF,OCB,2023-04-03,12,-1.0,-1.0,-1.0,-1.0,3.0,2.0


In [3]:
# Suppression des colonnes du triplon d'index 

X = X.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])
y = y.drop(columns=['Unnamed: 0'])

X.head()

,train,gare,date,arret,p2q0,p3q0,p4q0,p0q2,p0q3,p0q4
0,VBXNMF,KYF,2023-04-03,8,0.0,0.0,1.0,-3.0,-1.0,-2.0
1,VBXNMF,JLR,2023-04-03,9,0.0,0.0,0.0,1.0,0.0,1.0
2,VBXNMF,EOH,2023-04-03,10,-1.0,0.0,0.0,-1.0,0.0,0.0
3,VBXNMF,VXY,2023-04-03,11,-1.0,-1.0,0.0,2.0,-2.0,0.0
4,VBXNMF,OCB,2023-04-03,12,-1.0,-1.0,-1.0,-1.0,3.0,2.0


In [4]:
# Détection des données dupliquées
X.duplicated().sum()

np.int64(0)

In [5]:
# Détection des données vides ou incomplètes
pd.concat([X, y], axis = 1).isnull().any()


train    False
gare     False
date     False
arret    False
p2q0     False
p3q0     False
p4q0     False
p0q2     False
p0q3     False
p0q4     False
p0q0     False
dtype: bool

In [6]:
df_data = pd.concat([X, y], axis = 1)
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 667264 entries, 0 to 667263
Data columns (total 11 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   train   667264 non-null  object 
 1   gare    667264 non-null  object 
 2   date    667264 non-null  object 
 3   arret   667264 non-null  int64  
 4   p2q0    667264 non-null  float64
 5   p3q0    667264 non-null  float64
 6   p4q0    667264 non-null  float64
 7   p0q2    667264 non-null  float64
 8   p0q3    667264 non-null  float64
 9   p0q4    667264 non-null  float64
 10  p0q0    667264 non-null  float64
dtypes: float64(7), int64(1), object(3)
memory usage: 56.0+ MB


In [8]:
# Donner à la variable date un format de date pandas
X['date'] = pd.to_datetime(X['date'], format="%Y-%m-%d")

# Extraire les features temporels
X["year"] = X["date"].dt.year
X["month"] = X["date"].dt.month
X["day"] = X["date"].dt.day
X["weekday"] = X["date"].dt.weekday
X["week"] = X["date"].dt.isocalendar().week.astype(int)
X["dayofyear"] = X["date"].dt.dayofyear

# Donner une dimension cyclique aux features
X["month_sin"] = np.sin(2 * np.pi * X["month"] / 12)
X["month_cos"] = np.cos(2 * np.pi * X["month"] / 12)

X["weekday_sin"] = np.sin(2 * np.pi * X["weekday"] / 7)
X["weekday_cos"] = np.cos(2 * np.pi * X["weekday"] / 7)

X["dayofyear_sin"] = np.sin(2 * np.pi * X["dayofyear"] / 365)
X["dayofyear_cos"] = np.cos(2 * np.pi * X["dayofyear"] / 365)

# Optionnel : retirer la date brute
X = X.drop(columns=["date"])

In [9]:
from sklearn import model_selection

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, train_size = 0.7)

In [10]:
X_train["train"].unique() # Beaucoup de catégories différentes -> binary encoding pour éviter d'avoir trop de features


array(['FJNSTM', 'CZAGCL', 'JLALRO', ..., 'JCKCAN', 'RBMHQR', 'PTFZSK'],
      shape=(37405,), dtype=object)

In [11]:
from category_encoders.binary import BinaryEncoder

# Encoder
encoder_train = BinaryEncoder(cols=["train"])

# Fit sur train
X_train = encoder_train.fit_transform(X_train)

# Transform sur test
X_test = encoder_train.transform(X_test)



In [12]:
# Encoder
encoder_gare = BinaryEncoder(cols=["gare"])

# Fit sur train
X_train = encoder_gare.fit_transform(X_train)

# Transform sur test
X_test = encoder_gare.transform(X_test)


In [23]:
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm

n_estimators = 200

clf = RandomForestClassifier(
    max_depth=7,
    n_estimators=1,
    warm_start=True,  # permet d'ajouter des arbres petit à petit
    random_state=42
)

# Entraînement progressif
for i in tqdm(range(1, n_estimators + 1), desc="Training RandomForest"):
    clf.n_estimators = i
    clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)


Training RandomForest:   0%|          | 0/200 [00:00<?, ?it/s]

c:\Users\morga\Documents\code\data_challenge\sncf_transilien_2025\ds_env\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
Training RandomForest:   0%|          | 1/200 [00:01<05:52,  1.77s/it]c:\Users\morga\Documents\code\data_challenge\sncf_transilien_2025\ds_env\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
Training RandomForest:   1%|          | 2/200 [00:02<04:34,  1.39s/it]c:\Users\morga\Documents\code\data_challenge\sncf_transilien_2025\ds_env\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Prédictions
y_pred = clf.predict(X_test)

# MAE -> benchmark : 0.8930, meilleur actuel : 0.6346 sur X_test
mae = mean_absolute_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R2 score
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE :", rmse)
print("R2 :", r2)

MAE : 0.7857927864921571
RMSE : 1.772259345336224
R2 : 0.0734344649126547
